In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [29]:
data = pd.read_csv('churn_modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [30]:
data = data.drop(['RowNumber','CustomerId','Surname'], axis = 1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [31]:
hp_label_encoder_gender = LabelEncoder()
data['Gender'] = hp_label_encoder_gender.fit_transform(data['Gender'])

In [32]:
hp_one_hot_Encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = hp_one_hot_Encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=hp_one_hot_Encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [33]:
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [34]:
X = data.drop('Exited', axis = 1)
Y = data['Exited']

In [35]:
# split the data into training and testing set
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)   

## Scale these features
hp_scaler = StandardScaler()
X_train = hp_scaler.fit_transform(X_train)
X_test = hp_scaler.transform(X_test)

In [36]:
with open('hp_scaler.pkl', 'wb') as f:
    pickle.dump(hp_scaler, f)

## Save the encoders and scaler for future use
with open('hp_label_enc_gender.pkl', 'wb') as f:
    pickle.dump(hp_label_encoder_gender, f)

with open('hp_one_hot_encoder_geography.pkl', 'wb') as f:
    pickle.dump(hp_one_hot_Encoder_geo, f)

In [46]:
## define a function to create the model and try different hyperparameters(keras classifier)

def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))
    
    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))
    
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

In [48]:
## Create a keras classifier
model = KerasClassifier(layers=1, neurons=32, build_fn=create_model, verbose=1)

In [49]:
#define grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2, 3],
    # 'batch_size': [10],
    'epochs': [50, 100]
}

In [50]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3, verbose=1)
grid_result = grid.fit(X_train, Y_train)

print(f"Best: {grid_result.best_score_} using {grid_result.best_params_}")

Fitting 3 folds for each of 24 candidates, totalling 72 fits




c:\LearningWS\MachineLearning\mlenv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


Epoch 1/50


250/250 [==============================] - 1s 919us/step - loss: 0.4692 - accuracy: 0.7835
Epoch 2/50
250/250 [==============================] - 0s 755us/step - loss: 0.3946 - accuracy: 0.8353
Epoch 3/50
250/250 [==============================] - 0s 801us/step - loss: 0.3680 - accuracy: 0.8481
Epoch 4/50
250/250 [==============================] - 0s 809us/step - loss: 0.3548 - accuracy: 0.8549
Epoch 5/50
250/250 [==============================] - 0s 772us/step - loss: 0.3489 - accuracy: 0.8558
Epoch 6/50
250/250 [==============================] - 0s 805us/step - loss: 0.3460 - accuracy: 0.8575
Epoch 7/50
250/250 [==============================] - 0s 824us/step - loss: 0.3439 - accuracy: 0.8599
Epoch 8/50
250/250 [==============================] - 0s 848us/step - loss: 0.3408 - accuracy: 0.8604
Epoch 9/50
250/250 [==============================] - 0s 821us/step - loss: 0.3391 - accuracy: 0.8605
Epoch 10/50
250/250 [==============================] - 0s 878us/step - loss: 0.3